In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/telecom_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)


In [ ]:
# Q6: Churn rate by Gender, Senior Citizen, Partner, Dependents
churn_dimensions = ['gender', 'SeniorCitizen', 'Partner', 'Dependents']

results = []
for dim in churn_dimensions:
    group = df.groupby(dim)['Churn'].apply(
        lambda x: (x == 'Yes').sum() / len(x) * 100  # count how many churned in this group
    ).reset_index() # groupby returns a messy index. This converts it back to a clean regular table.
    group.columns = ['value', 'churn_rate'] # Rename the columns
    group['dimension'] = dim  # label which dimension
    results.append(group)

report = pd.concat(results).sort_values('churn_rate', ascending=False)
report['churn_rate'] = report['churn_rate'].round(2)
report[['dimension', 'value', 'churn_rate']]


,dimension,value,churn_rate
1,SeniorCitizen,1,41.68
0,Partner,No,32.96
0,Dependents,No,31.28
0,gender,Female,26.92
1,gender,Male,26.16
0,SeniorCitizen,0,23.61
1,Partner,Yes,19.66
1,Dependents,Yes,15.45


In [ ]:
# Q7: Customer segments based on tenure and churn analysis
df['tenure_segment'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72], # define segment boundaries
    labels=['New (0-12)', 'Growing (12-24)', 'Mature (24-48)', 'Loyal (48-72)']
)

# churn rate per segment
segment_analysis = df.groupby('tenure_segment').agg(
    customer_count = ('customerID', 'count'),   # customers per segment
    churned        = ('Churn', lambda x: (x == 'Yes').sum()), # churned customers
    churn_rate     = ('Churn', lambda x: round((x == 'Yes').sum() / len(x) * 100, 2))  # churn %
).reset_index()

segment_analysis


/var/folders/vl/k_d9mqcj5tx2z8pf5wfs_ss1ff99n6/T/ipykernel_88163/832323584.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  segment_analysis = df.groupby('tenure_segment').agg(


,tenure_segment,customer_count,churned,churn_rate
0,New (0-12),2175,1037,47.68
1,Growing (12-24),1024,294,28.71
2,Mature (24-48),1594,325,20.39
3,Loyal (48-72),2239,213,9.51


In [ ]:
# Q8: Top 10 customer profiles most likely to churn
# combination of Contract, PaymentMethod, InternetService
profiles = df.groupby(['Contract', 'PaymentMethod', 'InternetService']).agg(
    customer_count = ('customerID', 'count'),  # customers per profile
    churned = ('Churn', lambda x: (x == 'Yes').sum()), # churned per profile
    churn_rate = ('Churn', lambda x: round((x == 'Yes').sum() / len(x) * 100, 2)) # churn %
).reset_index()

# filter profiles with at least 30 customers to avoid small sample bias
profiles = profiles[profiles['customer_count'] >= 30]

# sort by churn rate and show top 10
profiles.sort_values('churn_rate', ascending=False).head(10)


,Contract,PaymentMethod,InternetService,customer_count,churned,churn_rate
7,Month-to-month,Electronic check,Fiber optic,1307,789,60.37
10,Month-to-month,Mailed check,Fiber optic,201,102,50.75
1,Month-to-month,Bank transfer (automatic),Fiber optic,327,149,45.57
4,Month-to-month,Credit card (automatic),Fiber optic,293,122,41.64
6,Month-to-month,Electronic check,DSL,474,192,40.51
9,Month-to-month,Mailed check,DSL,367,113,30.79
3,Month-to-month,Credit card (automatic),DSL,185,50,27.03
19,One year,Electronic check,Fiber optic,196,51,26.02
11,Month-to-month,Mailed check,No,325,67,20.62
2,Month-to-month,Bank transfer (automatic),No,65,13,20.00


In [ ]:
# Q9: Calculate CLV and compare churned vs retained customers
df['CLV'] = df['MonthlyCharges'] * df['tenure']   # lifetime value formula

# compare churned vs retained
clv_comparison = df.groupby('Churn').agg(
    customer_count = ('customerID', 'count'), # number of customers
    avg_clv = ('CLV', lambda x: round(x.mean(), 2)), # average CLV
    avg_tenure = ('tenure', lambda x: round(x.mean(), 2)), # average tenure
    avg_monthly = ('MonthlyCharges', lambda x: round(x.mean(), 2)) # average monthly charge
).reset_index()

clv_comparison


,Churn,customer_count,avg_clv,avg_tenure,avg_monthly
0,No,5174,2549.77,37.57,61.27
1,Yes,1869,1531.61,17.98,74.44


In [ ]:
# Q10: Customer Risk Score using Contract, Tenure, Tech Support, Online Security
def calculate_risk_score(row):
    score = 0

    # contract type — month-to-month is highest risk
    if row['Contract'] == 'Month-to-month':
        score += 3
    elif row['Contract'] == 'One year':
        score += 1
    else:   # Two year = lowest risk
        score += 0

    # tenure — newer customers are higher risk
    if row['tenure'] <= 12:
        score += 3
    elif row['tenure'] <= 24:
        score += 2
    elif row['tenure'] <= 48:
        score += 1
    else:
        score += 0

    # no tech support = higher risk
    if row['TechSupport'] == 'No':
        score += 2

    # no online security = higher risk
    if row['OnlineSecurity'] == 'No':
        score += 2

    return score

df['risk_score'] = df.apply(calculate_risk_score, axis=1)  # Runs the function on every row one at a time

# summarize risk score vs churn
df.groupby('risk_score')['Churn'].apply(
    lambda x: round((x == 'Yes').sum() / len(x) * 100, 2)
).reset_index().rename(columns={'Churn': 'churn_rate'})


,risk_score,churn_rate
0,0,2.39
1,1,4.33
2,2,4.74
3,3,8.76
4,4,10.40
5,5,15.42
6,6,24.85
7,7,36.41
8,8,41.02
9,9,45.96
